# Fake News Detector

In [45]:
import pandas as pd
import streamlit as st
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import string
import re
import pickle
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

In [46]:
data_fake=pd.read_csv('manual_testing.csv')
data_true=pd.read_csv('manual_testing.csv')

### Data Preview 

In [47]:
data_fake.head()
data_true.head()

,Unnamed: 0,title,text,subject,date,class
0,23471,Seven Iranians freed in the prisoner swap have...,"21st Century Wire says This week, the historic...",Middle-east,"January 20, 2016",0
1,23472,#Hashtag Hell & The Fake Left,By Dady Chery and Gilbert MercierAll writers ...,Middle-east,"January 19, 2016",0
2,23473,Astroturfing: Journalist Reveals Brainwashing ...,Vic Bishop Waking TimesOur reality is carefull...,Middle-east,"January 19, 2016",0
3,23474,The New American Century: An Era of Fraud,Paul Craig RobertsIn the last years of the 20t...,Middle-east,"January 19, 2016",0
4,23475,Hillary Clinton: ‘Israel First’ (and no peace ...,Robert Fantina CounterpunchAlthough the United...,Middle-east,"January 18, 2016",0


In [48]:
data_fake["class"]=0
data_true["class"]=1

In [49]:
data_fake.shape, data_true.shape

((20, 6), (20, 6))

In [50]:
# For data_fake
data_fake = data_fake.reset_index(drop=True)
data_fake_manual_testing = data_fake.tail(10)
for i in range(len(data_fake)-1, len(data_fake)-11, -1):
    data_fake.drop([i], axis=0, inplace=True)

# For data_true
data_true = data_true.reset_index(drop=True)
data_true_manual_testing = data_true.tail(10)
for i in range(len(data_true)-1, len(data_true)-11, -1):
    data_true.drop([i], axis=0, inplace=True)

In [51]:
data_fake.shape, data_true.shape

((10, 6), (10, 6))

In [52]:
data_fake_manual_testing['class']=0
data_true_manual_testing['class']=1

In [53]:
data_fake_manual_testing.head(10)

,Unnamed: 0,title,text,subject,date,class
10,21407,"Mata Pires, owner of embattled Brazil builder ...","SAO PAULO (Reuters) - Cesar Mata Pires, the ow...",worldnews,"August 22, 2017",0
11,21408,"U.S., North Korea clash at U.N. forum over nuc...",GENEVA (Reuters) - North Korea and the United ...,worldnews,"August 22, 2017",0
12,21409,"U.S., North Korea clash at U.N. arms forum on ...",GENEVA (Reuters) - North Korea and the United ...,worldnews,"August 22, 2017",0
13,21410,Headless torso could belong to submarine journ...,COPENHAGEN (Reuters) - Danish police said on T...,worldnews,"August 22, 2017",0
14,21411,North Korea shipments to Syria chemical arms a...,UNITED NATIONS (Reuters) - Two North Korean sh...,worldnews,"August 21, 2017",0
15,21412,'Fully committed' NATO backs new U.S. approach...,BRUSSELS (Reuters) - NATO allies on Tuesday we...,worldnews,"August 22, 2017",0
16,21413,LexisNexis withdrew two products from Chinese ...,"LONDON (Reuters) - LexisNexis, a provider of l...",worldnews,"August 22, 2017",0
17,21414,Minsk cultural hub becomes haven from authorities,MINSK (Reuters) - In the shadow of disused Sov...,worldnews,"August 22, 2017",0
18,21415,Vatican upbeat on possibility of Pope Francis ...,MOSCOW (Reuters) - Vatican Secretary of State ...,worldnews,"August 22, 2017",0
19,21416,Indonesia to buy $1.14 billion worth of Russia...,JAKARTA (Reuters) - Indonesia will buy 11 Sukh...,worldnews,"August 22, 2017",0


In [54]:
data_true_manual_testing.head(10)

,Unnamed: 0,title,text,subject,date,class
10,21407,"Mata Pires, owner of embattled Brazil builder ...","SAO PAULO (Reuters) - Cesar Mata Pires, the ow...",worldnews,"August 22, 2017",1
11,21408,"U.S., North Korea clash at U.N. forum over nuc...",GENEVA (Reuters) - North Korea and the United ...,worldnews,"August 22, 2017",1
12,21409,"U.S., North Korea clash at U.N. arms forum on ...",GENEVA (Reuters) - North Korea and the United ...,worldnews,"August 22, 2017",1
13,21410,Headless torso could belong to submarine journ...,COPENHAGEN (Reuters) - Danish police said on T...,worldnews,"August 22, 2017",1
14,21411,North Korea shipments to Syria chemical arms a...,UNITED NATIONS (Reuters) - Two North Korean sh...,worldnews,"August 21, 2017",1
15,21412,'Fully committed' NATO backs new U.S. approach...,BRUSSELS (Reuters) - NATO allies on Tuesday we...,worldnews,"August 22, 2017",1
16,21413,LexisNexis withdrew two products from Chinese ...,"LONDON (Reuters) - LexisNexis, a provider of l...",worldnews,"August 22, 2017",1
17,21414,Minsk cultural hub becomes haven from authorities,MINSK (Reuters) - In the shadow of disused Sov...,worldnews,"August 22, 2017",1
18,21415,Vatican upbeat on possibility of Pope Francis ...,MOSCOW (Reuters) - Vatican Secretary of State ...,worldnews,"August 22, 2017",1
19,21416,Indonesia to buy $1.14 billion worth of Russia...,JAKARTA (Reuters) - Indonesia will buy 11 Sukh...,worldnews,"August 22, 2017",1


In [55]:
data_merge=pd.concat([data_fake, data_true], axis = 0)
data_merge.head(10)

,Unnamed: 0,title,text,subject,date,class
0,23471,Seven Iranians freed in the prisoner swap have...,"21st Century Wire says This week, the historic...",Middle-east,"January 20, 2016",0
1,23472,#Hashtag Hell & The Fake Left,By Dady Chery and Gilbert MercierAll writers ...,Middle-east,"January 19, 2016",0
2,23473,Astroturfing: Journalist Reveals Brainwashing ...,Vic Bishop Waking TimesOur reality is carefull...,Middle-east,"January 19, 2016",0
3,23474,The New American Century: An Era of Fraud,Paul Craig RobertsIn the last years of the 20t...,Middle-east,"January 19, 2016",0
4,23475,Hillary Clinton: ‘Israel First’ (and no peace ...,Robert Fantina CounterpunchAlthough the United...,Middle-east,"January 18, 2016",0
5,23476,McPain: John McCain Furious That Iran Treated ...,21st Century Wire says As 21WIRE reported earl...,Middle-east,"January 16, 2016",0
6,23477,JUSTICE? Yahoo Settles E-mail Privacy Class-ac...,21st Century Wire says It s a familiar theme. ...,Middle-east,"January 16, 2016",0
7,23478,Sunnistan: US and Allied ‘Safe Zone’ Plan to T...,Patrick Henningsen 21st Century WireRemember ...,Middle-east,"January 15, 2016",0
8,23479,How to Blow $700 Million: Al Jazeera America F...,21st Century Wire says Al Jazeera America will...,Middle-east,"January 14, 2016",0
9,23480,10 U.S. Navy Sailors Held by Iranian Military ...,21st Century Wire says As 21WIRE predicted in ...,Middle-east,"January 12, 2016",0


In [56]:
data_merge.columns

Index(['Unnamed: 0', 'title', 'text', 'subject', 'date', 'class'], dtype='object')

In [57]:
data=data_merge.drop(['title','subject','date'], axis = 1)

In [58]:
#count of missing values
data.isnull().sum() 

Unnamed: 0    0
text          0
class         0
dtype: int64

In [59]:
data = data.sample(frac = 1)

In [60]:
data.head()

,Unnamed: 0,text,class
9,23480,21st Century Wire says As 21WIRE predicted in ...,1
5,23476,21st Century Wire says As 21WIRE reported earl...,0
2,23473,Vic Bishop Waking TimesOur reality is carefull...,1
1,23472,By Dady Chery and Gilbert MercierAll writers ...,1
9,23480,21st Century Wire says As 21WIRE predicted in ...,0


In [61]:
data.reset_index(inplace = True)
data.drop(['index'], axis = 1, inplace = True)

In [62]:
data.columns

Index(['Unnamed: 0', 'text', 'class'], dtype='object')

In [63]:
data.head()

,Unnamed: 0,text,class
0,23480,21st Century Wire says As 21WIRE predicted in ...,1
1,23476,21st Century Wire says As 21WIRE reported earl...,0
2,23473,Vic Bishop Waking TimesOur reality is carefull...,1
3,23472,By Dady Chery and Gilbert MercierAll writers ...,1
4,23480,21st Century Wire says As 21WIRE predicted in ...,0


## Preprocessing Text

#### Creating a function to convert the text in lowercase, remove the extra space, special chr., ulr and links.

In [64]:
def wordopt(text):
    text = text.lower()
    text = re.sub('\[.*?\]','',text)
    text = re.sub("\\W"," ",text)
    text = re.sub('https?://\S+|www\.\S+','',text)
    text = re.sub('<.*?>+',b'',text)
    text = re.sub('[%s]' % re.escape(string.punctuation),'',text)
    text = re.sub('\w*\d\w*','',text)
    return text

In [65]:
data['text'] = data['text'].apply(wordopt)

In [66]:
x = data['text']
y = data['class']

## Training the model

In [67]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size = 0.25)

In [68]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorization = TfidfVectorizer()
xv_train = vectorization.fit_transform(x_train)
xv_test = vectorization.transform(x_test)

## Logistic Regression

In [69]:
from sklearn.linear_model import LogisticRegression

In [70]:
LR = LogisticRegression()
LR.fit(xv_train, y_train)

LogisticRegression()

In [71]:
pred_lr = LR.predict(xv_test)

In [72]:
LR.score(xv_test, y_test)

0.2

In [73]:
print (classification_report(y_test, pred_lr))

              precision    recall  f1-score   support

           0       0.25      0.50      0.33         2
           1       0.00      0.00      0.00         3

    accuracy                           0.20         5
   macro avg       0.12      0.25      0.17         5
weighted avg       0.10      0.20      0.13         5



## Decision Tree Classifier

In [74]:
from sklearn.tree import DecisionTreeClassifier

DT = DecisionTreeClassifier()
DT.fit(xv_train, y_train)

DecisionTreeClassifier()

In [75]:
pred_dt = DT.predict(xv_test)

In [76]:
DT.score(xv_test, y_test)

0.2

In [77]:
print (classification_report(y_test, pred_lr))

              precision    recall  f1-score   support

           0       0.25      0.50      0.33         2
           1       0.00      0.00      0.00         3

    accuracy                           0.20         5
   macro avg       0.12      0.25      0.17         5
weighted avg       0.10      0.20      0.13         5



## Gradient Boost Classifier

In [78]:
from sklearn.ensemble import GradientBoostingClassifier
GB = GradientBoostingClassifier(random_state = 0)
GB.fit(xv_train, y_train)

GradientBoostingClassifier(random_state=0)

In [79]:
pred_gb = GB.predict(xv_test)

In [80]:
GB.score(xv_test, y_test)

0.2

In [81]:
print(classification_report(y_test, pred_gb))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.33      0.33      0.33         3

    accuracy                           0.20         5
   macro avg       0.17      0.17      0.17         5
weighted avg       0.20      0.20      0.20         5



## Random Forest Classifier

In [82]:
from sklearn.ensemble import RandomForestClassifier

RF = RandomForestClassifier(random_state = 0)
RF.fit(xv_train, y_train)

RandomForestClassifier(random_state=0)

In [83]:
pred_rf = RF.predict(xv_test)

In [84]:
RF.score(xv_test, y_test)

0.2

In [85]:
print (classification_report(y_test, pred_rf))

              precision    recall  f1-score   support

           0       0.25      0.50      0.33         2
           1       0.00      0.00      0.00         3

    accuracy                           0.20         5
   macro avg       0.12      0.25      0.17         5
weighted avg       0.10      0.20      0.13         5



## Testing the Model

In [86]:
def output_lable(n):
    if n==0:
        return "Fake News"
    elif n==1:
        return "Not A Fake News"
    
def manual_testing(news):
    testing_news = {"text":[news]}
    new_def_test = pd.DataFrame(testing_news)
    new_def_test['text'] = new_def_test["text"].apply(wordopt)
    new_x_test = new_def_test["text"]
    new_xv_test = vectorization.transform(new_x_test)
    pred_LR = LR.predict(new_xv_test)
    pred_DT = DT.predict(new_xv_test)
    pred_GB = GB.predict(new_xv_test)
    pred_RF = RF.predict(new_xv_test)
    
    return print("\n\nLR Predicition: {} \nDT Prediction: {} \nGBC Prediction: {} \nRFC Prediction:{}".format(output_lable(pred_LR[0]),
                                                                                                             output_lable(pred_DT[0]),
                                                                                                             output_lable(pred_GB[0]),
                                                                                                             output_lable(pred_RF[0])))

In [87]:
pickle.dump(LR, open('lr_model.pkl', 'wb'))
pickle.dump(DT, open('dt_model.pkl', 'wb'))
pickle.dump(GB, open('gb_model.pkl', 'wb'))
pickle.dump(RF, open('rf_model.pkl', 'wb'))
pickle.dump(vectorization, open('vectorizer.pkl', 'wb'))

In [88]:
def output_lable(n):
    if n == 0:
        return "Fake News"
    elif n == 1:
        return "Not A Fake News"

# Function to predict on new input
def manual_testing(news):
    # Load the models and vectorizer
    LR = pickle.load(open('lr_model.pkl', 'rb'))
    DT = pickle.load(open('dt_model.pkl', 'rb'))
    GB = pickle.load(open('gb_model.pkl', 'rb'))
    RF = pickle.load(open('rf_model.pkl', 'rb'))
    vectorization = pickle.load(open('vectorizer.pkl', 'rb'))

    testing_news = {"text": [news]}
    new_def_test = pd.DataFrame(testing_news)
    new_def_test['text'] = new_def_test["text"].apply(wordopt)
    new_x_test = new_def_test["text"]
    new_xv_test = vectorization.transform(new_x_test)
    pred_LR = LR.predict(new_xv_test)
    pred_DT = DT.predict(new_xv_test)
    pred_GB = GB.predict(new_xv_test)
    pred_RF = RF.predict(new_xv_test)

    return print("\n\nLR Prediction: {} \nDT Prediction: {} \nGBC Prediction: {} \nRFC Prediction: {}".format(output_lable(pred_LR[0]),
                                                                                                             output_lable(pred_DT[0]),
                                                                                                             output_lable(pred_GB[0]),
                                                                                                             output_lable(pred_RF[0])))

### Model Testing With Manual Entry

In [89]:
news = str(input()) 
manual_testing(news)



LR Prediction: Fake News 
DT Prediction: Fake News 
GBC Prediction: Not A Fake News 
RFC Prediction: Fake News


In [90]:
news=str(input())
manual_testing(news)



LR Prediction: Fake News 
DT Prediction: Fake News 
GBC Prediction: Not A Fake News 
RFC Prediction: Fake News


In [91]:
def main():
    st.title("Fake News Detection")
    news = st.text_area("Enter the news text here:")
    if st.button("Predict"):
        if news:
            result = manual_testing(news)
            st.write(result)
        else:
            st.warning("Please enter some text.")

if __name__ == '__main__':
    main()

2025-05-02 00:14:47.131 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-02 00:14:47.648 
  command:

    streamlit run C:\Users\ACER\AppData\Roaming\Python\Python311\site-packages\ipykernel_launcher.py [ARGUMENTS]
2025-05-02 00:14:47.651 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-02 00:14:47.653 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-02 00:14:47.655 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-02 00:14:47.659 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-02 00:14:47.662 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-02 00:14:47.664 Sessi

In [92]:
import pickle

with open('vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorization, f)